In [ ]:
import os
import pickle
import sys
import time

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio_ffmpeg
from joblib import Parallel, delayed, parallel_config

root_dir = os.path.abspath("..")
os.environ["PYTHONPATH"] = root_dir + os.pathsep + os.environ.get("PYTHONPATH", "")
data_dir = os.path.join(root_dir, "data")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.models.machines import ieee_machine2_trained_neural_flux_and_torque
from current_setpoints.models.forward_model import ForwardModel, Fault

CURR_MAX = 30.0
VOLT_MAX = 13.0
RUN_TAGS = ["12x10_N60_tauR_fix", "12x10_N60_tauR_fix_band2"]
VARIANT_TITLES = {"fix": "exact torque", "fix_band2": "torque band +-2 %"}
FPS = 4
HOLD_FRAMES = 4
DPI = 80
CASES = {
    "healthy": {"open": (), "title": "healthy",
                "pkl": "torque_speed_map_dynamic_neural_flux_torque_{tag}.pkl",
                "probe": "torque_speed_map_probe_neural_flux_torque_{tag}.pkl"},
    "one_fault": {"open": (0,), "title": "phase a open", "pkl": "fault_map_dynamic_one_fault_{tag}.pkl"},
    "two_adjacent": {"open": (0, 1), "title": "phases a, b open", "pkl": "fault_map_dynamic_two_adjacent_{tag}.pkl"},
    "two_nonadjacent": {"open": (0, 2), "title": "phases a, c open", "pkl": "fault_map_dynamic_two_nonadjacent_{tag}.pkl"},
}

In [ ]:
def fine_idx_grid(fwd, theta_grid):
    n_theta = fwd.vec_theta.size - 1
    return np.round(theta_grid / (2 * np.pi) * n_theta).astype(int) % n_theta


def alpha_beta_xy(theta_grid, sched):
    c1, s1 = np.cos(theta_grid), np.sin(theta_grid)
    c3, s3 = np.cos(3 * theta_grid), np.sin(3 * theta_grid)
    id1, iq1, id3, iq3 = sched[:, 0], sched[:, 1], sched[:, 2], sched[:, 3]
    i_alpha = c1 * id1 - s1 * iq1
    i_beta = s1 * id1 + c1 * iq1
    i_x = c3 * id3 - s3 * iq3
    i_y = s3 * id3 + c3 * iq3
    return i_alpha, i_beta, i_x, i_y


def waveforms_for_plot(fwd, omega, curr_dq, n_theta_plot=360):
    """(theta_deg, sched, iph, vph, theta_v_deg) for either a constant (static, curr_dq.ndim==1)
    or per-node-trajectory (dynamic, curr_dq.ndim==2) current, on a common convention suitable
    for the 9-panel diagnostic plot. Static is swept fresh over n_theta_plot samples of the
    SAME constant vector (current doesn't change over the cycle, only its phase reconstruction
    does, since the dq->phase map itself rotates with theta); dynamic reuses each node's own
    angle/current directly.

    Dynamic voltage: PDF eq. (7) with the current running linearly between nodes, i.e. WITH the
    dynamic term (omega/dtheta) L (i_next - i_n) -- two values per node, just before theta_n (end
    of the incoming segment) and just after it (start of the outgoing one), the same voltages
    ActiveSetOptimizer limits and grid.py's _trajectory_waveforms reports. vph then has 2 columns
    per node, plotted against theta_v_deg (each node angle twice)."""
    kept = list(fwd.fault._kept)
    dynamic = curr_dq.ndim == 2
    if not dynamic:
        theta_grid = np.linspace(0.0, 2 * np.pi, n_theta_plot, endpoint=False)
        sched = np.tile(curr_dq, (n_theta_plot, 1))
        idx_grid = fine_idx_grid(fwd, theta_grid)
        n_pts = n_theta_plot
    else:
        n_pts = curr_dq.shape[0]
        theta_grid = np.linspace(0.0, 2 * np.pi, n_pts, endpoint=False)
        idx_grid = fine_idx_grid(fwd, theta_grid)
        sched = curr_dq
    coupling = omega / (2 * np.pi / n_pts)

    iph = np.full((len(kept), n_pts), np.nan)
    vph = np.full((len(kept), 2 * n_pts if dynamic else n_pts), np.nan)
    for k, ti in enumerate(idx_grid):
        i_n = sched[k]
        if not np.all(np.isfinite(i_n)):
            continue
        iph[:, k] = fwd.phase_map_at_theta(int(ti)) @ i_n
        gU, gL, bV = fwd.volt_map_at_theta(int(ti), omega, i_n)
        v_steady = gU @ i_n + bV
        if dynamic:
            i_prev, i_next = sched[(k - 1) % n_pts], sched[(k + 1) % n_pts]
            vph[:, 2 * k] = (v_steady + coupling * (gL @ (i_n - i_prev)))[kept]
            vph[:, 2 * k + 1] = (v_steady + coupling * (gL @ (i_next - i_n)))[kept]
        else:
            vph[:, k] = v_steady[kept]
    theta_deg = np.rad2deg(theta_grid)
    theta_v_deg = np.repeat(theta_deg, 2) if dynamic else theta_deg
    return theta_deg, sched, iph, vph, theta_v_deg

In [ ]:
PHASE_LETTERS = ["a", "b", "c", "d", "e"]
DQ_LBL = [r"$i_{sd}^1$", r"$i_{sq}^1$", r"$i_{sd}^3$", r"$i_{sq}^3$"]
DQ_COL = ["tab:blue", "tab:orange", "tab:green", "tab:red"]


def point_data(fwd, omega, traj, torque):
    """Everything one frame shows, for one solved grid point (dynamic trajectory, shape (n_grid, dim))."""
    theta, sched, iph, vph, theta_v = waveforms_for_plot(fwd, omega, traj)
    i_alpha, i_beta, i_x, i_y = alpha_beta_xy(np.deg2rad(theta), sched)
    torque_pw = np.array([fwd.drive.torque(omega, sched[k]) for k in range(sched.shape[0])])
    spec = np.abs(np.fft.rfft(iph, axis=1)) / iph.shape[1]
    return {"theta": theta, "sched": sched, "iph": iph, "vph": vph, "theta_v": theta_v,
            "cu": np.sum(iph ** 2, axis=0), "ab": (i_alpha, i_beta), "xy": (i_x, i_y),
            "torque_pw": torque_pw, "spec": spec, "T": torque}


def span(values, sym=False, floor=None, pad=0.05):
    lo, hi = float(np.nanmin(values)), float(np.nanmax(values))
    if sym:
        m = max(abs(lo), abs(hi), floor or 0.0)
        return -m * (1 + pad), m * (1 + pad)
    d = max(hi - lo, 1e-9)
    return lo - pad * d, hi + pad * d


def speed_limits(points):
    """Axis ranges shared by all torque levels of one speed, so the axes stay put while the torque rises."""
    cat = lambda key: np.concatenate([np.ravel(p[key]) for p in points])
    ab = np.concatenate([np.ravel(p["ab"]) for p in points] + [np.ravel(p["xy"]) for p in points])
    return {"dq": span(cat("sched")), "iph": span(cat("iph"), sym=True, floor=CURR_MAX),
            "vph": span(cat("vph"), sym=True, floor=VOLT_MAX), "cu": (0.0, 1.05 * float(np.nanmax(cat("cu")))),
            "ab": span(ab, sym=True), "torque": span(np.concatenate([cat("torque_pw"), [p["T"] for p in points]])),
            "spec": (0.0, 1.05 * float(np.nanmax(cat("spec"))))}


def draw_frame(fig, fwd, case, variant, rpm, p, lim, map_pts, cur_pt, envelope):
    fig.clear()
    ax = fig.subplots(3, 3)
    letters = [PHASE_LETTERS[k] for k in fwd.fault._kept]
    th = p["theta"]
    fig.suptitle(f"Dynamic mode, {CASES[case]['title']}, {variant}:   {rpm:.0f} rpm,   T* = {p['T']:.2f} Nm", fontsize=15)

    a = ax[0, 0]
    for j in range(p["sched"].shape[1]):
        a.plot(th, p["sched"][:, j], color=DQ_COL[j], label=DQ_LBL[j])
    a.set_ylim(lim["dq"]); a.set_title("DQ setpoints"); a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("DQ current [A]")
    a.legend(fontsize=8, loc="upper right")

    a = ax[0, 1]
    for k, lt in enumerate(letters):
        a.plot(th, p["iph"][k], label=f"$i_{{{lt}}}$")
    a.axhline(CURR_MAX, color="gray", ls=":"); a.axhline(-CURR_MAX, color="gray", ls=":")
    a.set_ylim(lim["iph"]); a.set_title("Phase currents"); a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("phase current [A]")
    a.legend(fontsize=8, loc="upper right")

    a = ax[0, 2]
    for k, lt in enumerate(letters):
        a.plot(p["theta_v"], p["vph"][k], label=f"$v_{{{lt}}}$")
    a.axhline(VOLT_MAX, color="gray", ls=":"); a.axhline(-VOLT_MAX, color="gray", ls=":")
    a.set_ylim(lim["vph"]); a.set_title("Phase voltages"); a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("phase voltage [V]")
    a.legend(fontsize=8, loc="upper right")

    a = ax[1, 0]
    a.plot(th, p["cu"], color="tab:orange", label=f"mean={np.mean(p['cu']):.1f}")
    a.set_ylim(lim["cu"]); a.set_title(r"Joule-loss density $\sum_k i_k^2(\theta)$")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel(r"$\sum_k i_k^2$ [A$^2$]"); a.legend(fontsize=8, loc="upper right")

    a = ax[1, 1]
    if envelope is not None:
        a.plot(envelope[0], envelope[1], color="0.6", lw=1, label="max torque")
    a.scatter(map_pts[0], map_pts[1], s=8, c="0.65", marker=".", label="solved grid points")
    a.scatter([cur_pt[0]], [cur_pt[1]], s=140, c="tab:red", edgecolors="k", linewidths=0.6, zorder=3, label="shown point")
    a.set_xlim(map_pts[2]); a.set_ylim(map_pts[3])
    a.set_title("Torque-speed map"); a.set_xlabel("Speed [r/min]"); a.set_ylabel("Torque [Nm]"); a.grid(alpha=0.3)
    a.legend(fontsize=8, loc="upper right", markerscale=0.5)

    a = ax[1, 2]
    a.plot(*p["ab"], color="tab:purple")
    a.set_xlim(lim["ab"]); a.set_ylim(lim["ab"]); a.set_aspect("equal")
    a.set_title(r"$\alpha\beta$ current"); a.set_xlabel(r"$i_\alpha$ [A]"); a.set_ylabel(r"$i_\beta$ [A]")

    a = ax[2, 0]
    a.plot(*p["xy"], color="tab:brown")
    a.set_xlim(lim["ab"]); a.set_ylim(lim["ab"]); a.set_aspect("equal")
    a.set_title("xy current"); a.set_xlabel(r"$i_x$ [A]"); a.set_ylabel(r"$i_y$ [A]")

    a = ax[2, 1]
    a.plot(th, p["torque_pw"], color="tab:red", label=f"mean={np.mean(p['torque_pw']):.3f}")
    a.axhline(p["T"], color="gray", ls=":", label=f"T*={p['T']:.2f}")
    a.set_ylim(lim["torque"]); a.set_title("Achieved torque (per-node constraint check)")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("torque [Nm]"); a.legend(fontsize=8, loc="upper right")

    a = ax[2, 2]
    h = np.arange(p["spec"].shape[1])
    width = 0.8 / len(letters)
    for k, lt in enumerate(letters):
        a.bar(h + k * width, p["spec"][k], width=width, label=f"$i_{{{lt}}}$")
    a.set_xlim(-0.5, min(15, h[-1]) + 0.5); a.set_ylim(lim["spec"])
    a.set_title("Phase current spectrum (electrical harmonics)"); a.set_xlabel("harmonic order"); a.set_ylabel("amplitude [A]")
    a.legend(fontsize=8, loc="upper right")
    fig.tight_layout(rect=[0, 0, 1, 0.96])


def make_video(case, tag):
    cfg = CASES[case]
    variant = VARIANT_TITLES[tag.split("tauR_", 1)[1]]
    video_dir = os.path.join(root_dir, "figs", f"waveform_videos_{tag}")
    os.makedirs(video_dir, exist_ok=True)
    drive = ieee_machine2_trained_neural_flux_and_torque(curr_max=CURR_MAX, volt_max=VOLT_MAX)
    fwd = ForwardModel(drive, n_theta=700, fault=Fault(cfg["open"]))
    with open(os.path.join(data_dir, cfg["pkl"].format(tag=tag)), "rb") as f:
        grid = pickle.load(f)
    rpm_axis = grid["vec_omega"] * grid["const_mech_speed"]
    torq = grid["grid_torq_neural"]
    solved = np.isfinite(grid["grid_joule_loss"])
    traj = grid["curr_dq_traj"]
    if "probe" in cfg:
        with open(os.path.join(data_dir, cfg["probe"].format(tag=tag)), "rb") as f:
            envelope = (rpm_axis, pickle.load(f)["tmax_old_row"])
    else:
        envelope = (rpm_axis, grid["probe_tmax"])
    O2 = np.broadcast_to(rpm_axis[None, :], torq.shape)
    t_top = float(np.nanmax(torq[solved]))
    map_pts = (O2[solved], torq[solved], (-0.03 * rpm_axis[-1], 1.03 * rpm_axis[-1]),
               (min(-0.03 * t_top, float(np.nanmin(envelope[1])) - 0.03 * t_top), 1.06 * t_top))

    path = os.path.join(video_dir, f"waveforms_dynamic_{case}.mp4")
    fig = plt.figure(figsize=(18, 13), dpi=DPI)
    size = tuple(int(v) for v in fig.canvas.get_width_height())
    writer = imageio_ffmpeg.write_frames(path, size, fps=FPS, codec="libx264", pix_fmt_out="yuv420p",
                                         output_params=["-crf", "20"], macro_block_size=16)
    writer.send(None)
    n_frames = 0
    t0 = time.perf_counter()
    for io in range(len(rpm_axis)):
        rows = [it for it in range(torq.shape[0]) if solved[it, io]]
        rows.sort(key=lambda it: torq[it, io])
        if not rows:
            continue
        omega = float(grid["vec_omega"][io])
        points = [point_data(fwd, omega, traj[it, io], float(torq[it, io])) for it in rows]
        lim = speed_limits(points)
        for k, (it, p) in enumerate(zip(rows, points)):
            draw_frame(fig, fwd, case, variant, rpm_axis[io], p, lim, map_pts, (rpm_axis[io], torq[it, io]), envelope)
            fig.canvas.draw()
            frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3].copy()
            for _ in range(1 + (HOLD_FRAMES if k == 0 else 0)):
                writer.send(frame)
            n_frames += 1
    writer.close()
    plt.close(fig)
    return f"{tag} {case}: {n_frames} grid points, {time.perf_counter() - t0:.0f} s, {os.path.getsize(path) / 1e6:.1f} MB -> {path}"

In [ ]:
t0 = time.perf_counter()
with parallel_config(backend="loky", inner_max_num_threads=1):
    reports = Parallel(n_jobs=len(CASES) * len(RUN_TAGS))(delayed(make_video)(c, t) for t in RUN_TAGS for c in CASES)
for r in reports:
    print(r)
print(f"all videos done in {time.perf_counter() - t0:.0f} s")
